<a href="https://colab.research.google.com/github/PETEROA/ML-Optimization-Daily/blob/main/once_for_all.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Once-for-All Networks
Here I attempt to implement Once-for-All (OFA) networks, a paradigm where we train a single supernet once and deploy specialized subnets for different hardware constraints without retraining. Traditional NAS requires searching and training for each target device. OFA decouples training from search: train one elastic network that supports variable depth, width, kernel size, and resolution, then instantly extract optimized subnets for any hardware. We implement elastic convolutions, progressive shrinking training, subnet sampling, and accuracy predictors. OFA reduces the cost of deploying to N devices from O(N × training_cost) to O(1 × training_cost + N × search_cost). This is the culmination of efficient architecture design, combining NAS, hardware-awareness, and weight sharing into a practical deployment framework.

In [19]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
from typing import Optional, List, Dict, Tuple, Union
from dataclasses import dataclass, field
from collections import OrderedDict
import random
import copy
import time
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


Elastic Dimensions

Define the configuration space for elastic networks.

In [33]:
@dataclass
class ElasticConfig:
    """
    Configuration for elastic dimensions.

    Each dimension has a list of possible values.
    The supernet supports ALL combinations.
    """
    # Depth: number of blocks per stage
    # Reduced depth options to align with smaller supernet
    depth_options: List[int] = field(default_factory=lambda: [1, 2])

    # Width: channel expansion ratios
    width_options: List[float] = field(default_factory=lambda: [0.5, 0.75, 1.0])

    # Kernel size options
    kernel_options: List[int] = field(default_factory=lambda: [3, 5, 7])

    # Input resolution options
    resolution_options: List[int] = field(default_factory=lambda: [160, 192, 224])

    # Expand ratio options (for inverted residuals)
    expand_options: List[float] = field(default_factory=lambda: [3, 4, 6])

    def count_subnets(self, num_stages: int = 5, blocks_per_stage: int = 2) -> int:
        """
        Count total number of possible subnet configurations.
        (Adjusted blocks_per_stage for consistency with smaller supernet)
        """
        # Resolution choices
        res_choices = len(self.resolution_options)

        # For each stage: depth choices
        depth_choices = len(self.depth_options) ** num_stages

        # For each block: kernel and expand choices
        total_blocks = num_stages * blocks_per_stage # Using adjusted blocks_per_stage
        block_choices = (len(self.kernel_options) * len(self.expand_options)) ** total_blocks

        return res_choices * depth_choices * block_choices


@dataclass
class SubnetConfig:
    """
    Specific subnet configuration sampled from elastic space.
    """
    resolution: int = 224
    # Adjusted default depths to match smaller supernet
    depths: List[int] = field(default_factory=lambda: [2, 2, 2, 2, 2])
    widths: List[float] = field(default_factory=lambda: [1.0] * 5)
    kernel_sizes: List[List[int]] = None  # Per block per stage
    expand_ratios: List[List[float]] = None  # Per block per stage


# Test configuration space
elastic_config = ElasticConfig()
num_subnets = elastic_config.count_subnets(blocks_per_stage=2) # Adjusted blocks_per_stage for count
print(f"Elastic configuration space:")
print(f"  Depth options: {elastic_config.depth_options}")
print(f"  Width options: {elastic_config.width_options}")
print(f"  Kernel options: {elastic_config.kernel_options}")
print(f"  Resolution options: {elastic_config.resolution_options}")
print(f"\nTotal possible subnets: ~{num_subnets:.2e}")

Elastic configuration space:
  Depth options: [1, 2]
  Width options: [0.5, 0.75, 1.0]
  Kernel options: [3, 5, 7]
  Resolution options: [160, 192, 224]

Total possible subnets: ~3.35e+11


 Elastic Convolutions

The key building block: convolutions that can operate at different kernel sizes and channel widths using the SAME weights.

In [21]:
class ElasticConv2d(nn.Module):
    """
    Elastic Convolution that supports multiple kernel sizes.

    Key insight: A 7×7 kernel contains a 5×5 kernel contains a 3×3 kernel.
    We train the full 7×7 and extract smaller kernels from the center.

    Kernel transformation matrices ensure smooth transition between sizes.
    """

    def __init__(self, max_in_channels: int, max_out_channels: int,
                 kernel_sizes: List[int] = [3, 5, 7], stride: int = 1,
                 groups: int = 1, bias: bool = False):
        super().__init__()

        self.max_in_channels = max_in_channels
        self.max_out_channels = max_out_channels
        self.kernel_sizes = sorted(kernel_sizes)
        self.max_kernel = max(kernel_sizes)
        self.stride = stride
        self.groups = groups

        # Full-size convolution weights
        self.weight = nn.Parameter(torch.Tensor(
            max_out_channels, max_in_channels // groups,
            self.max_kernel, self.max_kernel
        ))

        if bias:
            self.bias = nn.Parameter(torch.Tensor(max_out_channels))
        else:
            self.register_parameter('bias', None)

        # Kernel transformation matrices for smooth interpolation
        self.kernel_transforms = self._build_transforms()

        self._init_weights()

    def _init_weights(self):
        nn.init.kaiming_normal_(self.weight, mode='fan_out', nonlinearity='relu')
        if self.bias is not None:
            nn.init.zeros_(self.bias)

    def _build_transforms(self) -> Dict[int, torch.Tensor]:
        """
        Build transformation matrices for kernel size reduction.

        These help smooth the transition between kernel sizes.
        """
        transforms = {}
        for ks in self.kernel_sizes[:-1]:  # All except largest
            # Simple center crop for now
            # In full OFA, this would be a learned transformation
            transforms[ks] = torch.ones(1)
        return transforms

    def get_active_kernel(self, kernel_size: int,
                          in_channels: int, out_channels: int) -> torch.Tensor:
        """
        Extract kernel of specified size from full weights.
        """
        # Get channel subset
        # For depthwise convolution (where groups == max_in_channels), the input channel dim is always 1.
        if self.groups == self.max_in_channels and self.groups > 1:
            weight = self.weight[:out_channels, :1]
        else:
            # For standard or grouped convolutions, calculate channels per group.
            weight = self.weight[:out_channels, :in_channels // self.groups]

        # Extract center kernel
        if kernel_size == self.max_kernel:
            return weight

        start = (self.max_kernel - kernel_size) // 2
        end = start + kernel_size
        return weight[:, :, start:end, start:end]

    def forward(self, x: torch.Tensor,
                kernel_size: Optional[int] = None,
                in_channels: Optional[int] = None,
                out_channels: Optional[int] = None) -> torch.Tensor:
        """
        Forward with specified elastic dimensions.

        Args:
            x: Input tensor
            kernel_size: Active kernel size (default: max)
            in_channels: Active input channels (default: max)
            out_channels: Active output channels (default: max)
        """
        kernel_size = kernel_size or self.max_kernel
        in_channels = in_channels or self.max_in_channels
        out_channels = out_channels or self.max_out_channels

        # Get active kernel
        weight = self.get_active_kernel(kernel_size, in_channels, out_channels)

        # Get active bias
        bias = self.bias[:out_channels] if self.bias is not None else None

        # Compute padding for same output size
        padding = kernel_size // 2

        # Adjust groups if needed
        groups = self.groups
        if self.groups == self.max_in_channels:  # Depthwise
            groups = in_channels

        return F.conv2d(x, weight, bias, self.stride, padding, groups=groups)

In [22]:
class ElasticBatchNorm2d(nn.Module):
    """
    Elastic BatchNorm that supports variable channel widths.

    Maintains separate BN statistics for different width configurations.
    """

    def __init__(self, max_channels: int):
        super().__init__()

        self.max_channels = max_channels
        self.bn = nn.BatchNorm2d(max_channels)

    def forward(self, x: torch.Tensor,
                channels: Optional[int] = None) -> torch.Tensor:
        """
        Forward with specified channel width.
        """
        channels = channels or self.max_channels

        if channels == self.max_channels:
            return self.bn(x)

        # Use subset of BN parameters
        weight = self.bn.weight[:channels]
        bias = self.bn.bias[:channels]
        running_mean = self.bn.running_mean[:channels]
        running_var = self.bn.running_var[:channels]

        return F.batch_norm(
            x, running_mean, running_var, weight, bias,
            self.training, self.bn.momentum, self.bn.eps
        )


class ElasticLinear(nn.Module):
    """
    Elastic Linear layer supporting variable input/output features.
    """

    def __init__(self, max_in_features: int, max_out_features: int, bias: bool = True):
        super().__init__()

        self.max_in_features = max_in_features
        self.max_out_features = max_out_features

        self.linear = nn.Linear(max_in_features, max_out_features, bias=bias)

    def forward(self, x: torch.Tensor,
                in_features: Optional[int] = None,
                out_features: Optional[int] = None) -> torch.Tensor:
        """
        Forward with specified dimensions.
        """
        in_features = in_features or self.max_in_features
        out_features = out_features or self.max_out_features

        weight = self.linear.weight[:out_features, :in_features]
        bias = self.linear.bias[:out_features] if self.linear.bias is not None else None

        return F.linear(x, weight, bias)

In [23]:
# Test elastic convolution

print("Testing Elastic Convolution")
print("=" * 50)

elastic_conv = ElasticConv2d(
    max_in_channels=64, max_out_channels=128,
    kernel_sizes=[3, 5, 7], stride=1
)

x = torch.randn(2, 64, 32, 32)

# Test different configurations
configs = [
    {'kernel_size': 7, 'in_channels': 64, 'out_channels': 128},
    {'kernel_size': 5, 'in_channels': 64, 'out_channels': 128},
    {'kernel_size': 3, 'in_channels': 64, 'out_channels': 128},
    {'kernel_size': 3, 'in_channels': 32, 'out_channels': 64},
]

for cfg in configs:
    x_sub = x[:, :cfg['in_channels']]
    y = elastic_conv(x_sub, **cfg)
    print(f"k={cfg['kernel_size']}, in={cfg['in_channels']}, out={cfg['out_channels']} "
          f"→ {y.shape}")

Testing Elastic Convolution
k=7, in=64, out=128 → torch.Size([2, 128, 32, 32])
k=5, in=64, out=128 → torch.Size([2, 128, 32, 32])
k=3, in=64, out=128 → torch.Size([2, 128, 32, 32])
k=3, in=32, out=64 → torch.Size([2, 64, 32, 32])


Elastic MobileNet Block

Building the elastic inverted residual block used in OFA-MobileNetV3.

In [24]:
class ElasticMBConv(nn.Module):
    """
    Elastic Mobile Inverted Bottleneck Conv (MBConv).

    Supports elastic:
    - Kernel size (3, 5, 7)
    - Expansion ratio (3, 4, 6)
    - Input/output channels
    """

    def __init__(self, max_in_channels: int, max_out_channels: int,
                 kernel_sizes: List[int] = [3, 5, 7],
                 expand_ratios: List[float] = [3, 4, 6],
                 stride: int = 1, use_se: bool = True):
        super().__init__()

        self.max_in_channels = max_in_channels
        self.max_out_channels = max_out_channels
        self.kernel_sizes = kernel_sizes
        self.expand_ratios = expand_ratios
        self.stride = stride
        self.use_se = use_se

        # Maximum expansion
        max_expand = max(expand_ratios)
        max_hidden = int(max_in_channels * max_expand)
        self.max_hidden = max_hidden

        # Expand: 1×1 conv
        self.expand_conv = ElasticConv2d(
            max_in_channels, max_hidden, kernel_sizes=[1]
        )
        self.expand_bn = ElasticBatchNorm2d(max_hidden)

        # Depthwise: elastic kernel
        self.depthwise_conv = ElasticConv2d(
            max_hidden, max_hidden, kernel_sizes=kernel_sizes,
            stride=stride, groups=max_hidden
        )
        self.depthwise_bn = ElasticBatchNorm2d(max_hidden)

        # SE block
        if use_se:
            se_channels = max(1, max_hidden // 4)
            self.se_pool = nn.AdaptiveAvgPool2d(1)
            self.se_fc1 = ElasticLinear(max_hidden, se_channels)
            self.se_fc2 = ElasticLinear(se_channels, max_hidden)

        # Project: 1×1 conv
        self.project_conv = ElasticConv2d(
            max_hidden, max_out_channels, kernel_sizes=[1]
        )
        self.project_bn = ElasticBatchNorm2d(max_out_channels)

        self.act = nn.SiLU(inplace=True)

    def forward(self, x: torch.Tensor,
                kernel_size: int = 7,
                expand_ratio: float = 6,
                in_channels: Optional[int] = None,
                out_channels: Optional[int] = None) -> torch.Tensor:
        """
        Forward with specified elastic configuration.
        """
        in_channels = in_channels or self.max_in_channels
        out_channels = out_channels or self.max_out_channels
        hidden_channels = int(in_channels * expand_ratio)

        # Use residual if dimensions match
        use_residual = (self.stride == 1 and in_channels == out_channels)
        identity = x if use_residual else None

        # Expand
        out = self.expand_conv(x, kernel_size=1,
                               in_channels=in_channels,
                               out_channels=hidden_channels)
        out = self.expand_bn(out, channels=hidden_channels)
        out = self.act(out)

        # Depthwise
        out = self.depthwise_conv(out, kernel_size=kernel_size,
                                   in_channels=hidden_channels,
                                   out_channels=hidden_channels)
        out = self.depthwise_bn(out, channels=hidden_channels)
        out = self.act(out)

        # SE
        if self.use_se:
            se = self.se_pool(out)
            se = se.view(se.size(0), -1)
            se_channels = max(1, hidden_channels // 4)
            se = self.act(self.se_fc1(se, in_features=hidden_channels,
                                       out_features=se_channels))
            se = torch.sigmoid(self.se_fc2(se, in_features=se_channels,
                                            out_features=hidden_channels))
            out = out * se.view(se.size(0), hidden_channels, 1, 1)

        # Project
        out = self.project_conv(out, kernel_size=1,
                                 in_channels=hidden_channels,
                                 out_channels=out_channels)
        out = self.project_bn(out, channels=out_channels)

        # Residual
        if use_residual:
            out = out + identity

        return out

In [25]:
# Test elastic MBConv

print("Testing Elastic MBConv")
print("=" * 50)

elastic_block = ElasticMBConv(
    max_in_channels=64, max_out_channels=64,
    kernel_sizes=[3, 5, 7], expand_ratios=[3, 4, 6]
)

x = torch.randn(2, 64, 16, 16)

configs = [
    {'kernel_size': 7, 'expand_ratio': 6},
    {'kernel_size': 5, 'expand_ratio': 4},
    {'kernel_size': 3, 'expand_ratio': 3},
]

for cfg in configs:
    y = elastic_block(x, **cfg)
    # Count active parameters
    hidden = int(64 * cfg['expand_ratio'])
    params = 64 * hidden + hidden * cfg['kernel_size']**2 + hidden * 64
    print(f"k={cfg['kernel_size']}, e={cfg['expand_ratio']} → {y.shape}, ~{params:,} params")

Testing Elastic MBConv
k=7, e=6 → torch.Size([2, 64, 16, 16]), ~67,968 params
k=5, e=4 → torch.Size([2, 64, 16, 16]), ~39,168 params
k=3, e=3 → torch.Size([2, 64, 16, 16]), ~26,304 params


Once-for-All Network

Complete OFA network with elastic depth, width, kernel size, and resolution.

In [32]:
class OFAMobileNetV3(nn.Module):
    """
    Once-for-All MobileNetV3.

    Supports elastic:
    - Resolution: handled by input size
    - Depth: number of blocks per stage
    - Width: channel multiplier
    - Kernel size: per block
    - Expand ratio: per block
    """

    def __init__(self, num_classes: int = 1000,
                 elastic_config: ElasticConfig = None):
        super().__init__()

        self.elastic_config = elastic_config or ElasticConfig()

        # Stage configurations: (out_channels, num_blocks, stride)
        # Reduced num_blocks from 4 to 2 to reduce supernet size
        self.stage_configs = [
            (24, 2, 2),   # Stage 1
            (40, 2, 2),   # Stage 2
            (80, 2, 2),   # Stage 3
            (112, 2, 1),  # Stage 4
            (160, 2, 2),  # Stage 5
        ]
        self.max_depth = 2 # Reduced max_depth to match stage configs

        # Stem
        self.stem = nn.Sequential(
            nn.Conv2d(3, 16, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.SiLU(inplace=True)
        )

        # First block (not elastic)
        self.first_block = ElasticMBConv(
            max_in_channels=16, max_out_channels=16,
            kernel_sizes=[3], expand_ratios=[1], stride=1, use_se=False
        )

        # Build elastic stages
        self.stages = nn.ModuleList()
        in_channels = 16

        for out_channels, num_blocks, stride in self.stage_configs:
            stage = nn.ModuleList()
            for i in range(num_blocks):
                block_stride = stride if i == 0 else 1
                block_in = in_channels if i == 0 else out_channels

                stage.append(ElasticMBConv(
                    max_in_channels=block_in,
                    max_out_channels=out_channels,
                    kernel_sizes=self.elastic_config.kernel_options,
                    expand_ratios=self.elastic_config.expand_options,
                    stride=block_stride,
                    use_se=True
                ))

            self.stages.append(stage)
            in_channels = out_channels

        # Head
        self.head_conv = nn.Sequential(
            nn.Conv2d(160, 960, 1, bias=False),
            nn.BatchNorm2d(960),
            nn.SiLU(inplace=True)
        )
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Linear(960, 1280),
            nn.SiLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(1280, num_classes)
        )

    def forward(self, x: torch.Tensor,
                subnet_config: SubnetConfig = None) -> torch.Tensor:
        """
        Forward pass with optional subnet configuration.

        Args:
            x: Input tensor
            subnet_config: Subnet configuration (None = use maximum)
        """
        # Default to maximum configuration
        if subnet_config is None:
            depths = [self.max_depth] * len(self.stages)
            kernel_sizes = [[7] * self.max_depth for _ in self.stages]
            expand_ratios = [[6] * self.max_depth for _ in self.stages]
        else:
            depths = subnet_config.depths
            kernel_sizes = subnet_config.kernel_sizes or \
                          [[7] * d for d in depths]
            expand_ratios = subnet_config.expand_ratios or \
                           [[6] * d for d in depths]

        # Stem
        out = self.stem(x)
        out = self.first_block(out, kernel_size=3, expand_ratio=1)

        # Stages with elastic depth
        for stage_idx, stage in enumerate(self.stages):
            depth = depths[stage_idx]
            for block_idx in range(depth):
                ks = kernel_sizes[stage_idx][block_idx]
                er = expand_ratios[stage_idx][block_idx]
                out = stage[block_idx](out, kernel_size=ks, expand_ratio=er)

        # Head
        out = self.head_conv(out)
        out = self.avgpool(out)
        out = out.view(out.size(0), -1)
        out = self.classifier(out)

        return out

    def sample_subnet(self) -> SubnetConfig:
        """
        Sample a random subnet configuration.
        """
        ec = self.elastic_config

        resolution = random.choice(ec.resolution_options)
        depths = [random.choice(ec.depth_options) for _ in self.stages]

        kernel_sizes = [
            [random.choice(ec.kernel_options) for _ in range(d)]
            for d in depths
        ]
        expand_ratios = [
            [random.choice(ec.expand_options) for _ in range(d)]
            for d in depths
        ]

        return SubnetConfig(
            resolution=resolution,
            depths=depths,
            kernel_sizes=kernel_sizes,
            expand_ratios=expand_ratios
        )

In [27]:
# Test OFA network

print("Testing Once-for-All Network")
print("=" * 60)

ofa_net = OFAMobileNetV3(num_classes=10)

# Test maximum configuration
x_max = torch.randn(2, 3, 224, 224)
y_max = ofa_net(x_max)
print(f"Maximum subnet: input {x_max.shape} → output {y_max.shape}")

# Test minimum configuration
min_config = SubnetConfig(
    resolution=160,
    depths=[2, 2, 2, 2, 2],
    kernel_sizes=[[3, 3], [3, 3], [3, 3], [3, 3], [3, 3]],
    expand_ratios=[[3, 3], [3, 3], [3, 3], [3, 3], [3, 3]]
)
x_min = torch.randn(2, 3, 160, 160)
y_min = ofa_net(x_min, min_config)
print(f"Minimum subnet: input {x_min.shape} → output {y_min.shape}")

# Sample random subnets
print("\nRandom subnet samples:")
for i in range(3):
    cfg = ofa_net.sample_subnet()
    total_blocks = sum(cfg.depths)
    avg_kernel = np.mean([np.mean(ks) for ks in cfg.kernel_sizes])
    avg_expand = np.mean([np.mean(er) for er in cfg.expand_ratios])
    print(f"  Sample {i+1}: res={cfg.resolution}, blocks={total_blocks}, "
          f"avg_k={avg_kernel:.1f}, avg_e={avg_expand:.1f}")

Testing Once-for-All Network
Maximum subnet: input torch.Size([2, 3, 224, 224]) → output torch.Size([2, 10])
Minimum subnet: input torch.Size([2, 3, 160, 160]) → output torch.Size([2, 10])

Random subnet samples:
  Sample 1: res=224, blocks=13, avg_k=4.4, avg_e=4.6
  Sample 2: res=192, blocks=15, avg_k=4.2, avg_e=4.7
  Sample 3: res=160, blocks=13, avg_k=5.2, avg_e=4.5


Progressive Shrinking

The key training technique: start with the largest subnet, then progressively enable smaller subnets.

In [41]:
class ProgressiveShrinkingTrainer:
    """
    Progressive Shrinking training for OFA networks.

    Training phases:
    1. Train full network (largest configuration)
    2. Enable elastic kernel sizes (7→5→3)
    3. Enable elastic depth (4→3→2)
    4. Enable elastic expand ratios (6→4→3)

    Each phase uses knowledge distillation from larger subnets.
    """

    def __init__(self, model: OFAMobileNetV3,
                 train_loader: DataLoader,
                 val_loader: DataLoader,
                 lr: float = 0.01):
        super().__init__()

        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader

        self.optimizer = torch.optim.SGD(
            model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4
        )

        # Update phase_configs to align with the new model's max_depth (2)
        # and depth_options ([1, 2])
        self.phase_configs = {
            'full': {
                'kernel_sizes': [7],
                'depths': [self.model.max_depth], # Use model's max_depth
                'expand_ratios': [6],
            },
            'elastic_kernel': {
                'kernel_sizes': [3, 5, 7],
                'depths': [self.model.max_depth], # Use model's max_depth
                'expand_ratios': [6],
            },
            'elastic_depth': {
                'kernel_sizes': [3, 5, 7],
                'depths': self.model.elastic_config.depth_options, # Use model's depth_options
                'expand_ratios': [6],
            },
            'elastic_expand': {
                'kernel_sizes': [3, 5, 7],
                'depths': self.model.elastic_config.depth_options, # Use model's depth_options
                'expand_ratios': [3, 4, 6],
            },
        }

        self.history = []

    def _sample_subnet_for_phase(self, phase: str) -> SubnetConfig:
        """
        Sample subnet configuration for current training phase.
        """
        cfg = self.phase_configs[phase]
        ec = self.model.elastic_config

        resolution = random.choice(ec.resolution_options)
        # Ensure sampled depths respect the model's actual stage structure
        depths = [random.choice(cfg['depths']) for _ in self.model.stages] # Use model.stages for length

        kernel_sizes = [
            [random.choice(cfg['kernel_sizes']) for _ in range(d)]
            for d in depths
        ]
        expand_ratios = [
            [random.choice(cfg['expand_ratios']) for _ in range(d)]
            for d in depths
        ]

        return SubnetConfig(
            resolution=resolution,
            depths=depths,
            kernel_sizes=kernel_sizes,
            expand_ratios=expand_ratios
        )

    def train_phase(self, phase: str, epochs: int = 5,
                    subnets_per_batch: int = 4) -> Dict:
        """
        Train one phase of progressive shrinking.
        """
        print(f"\n{'='*50}")
        print(f"Phase: {phase}")
        print(f"  Kernels: {self.phase_configs[phase]['kernel_sizes']}")
        print(f"  Depths: {self.phase_configs[phase]['depths']}")
        print(f"  Expands: {self.phase_configs[phase]['expand_ratios']}")
        print(f"{'='*50}")

        # self.model.to(device) # Moved this to the model instantiation in _VReMix8jflm

        for epoch in range(epochs):
            self.model.train()
            total_loss = 0

            for batch_idx, (data, target) in enumerate(self.train_loader):
                data, target = data.to(device), target.to(device)

                self.optimizer.zero_grad()
                batch_loss = 0

                # Train multiple subnets per batch
                teacher_output = None
                for i in range(subnets_per_batch):
                    # Sample subnet (largest first for teacher)
                    if i == 0:
                        # Largest subnet as teacher, respecting current model's max_depth
                        subnet_cfg = SubnetConfig(
                            resolution=data.shape[-1],
                            depths=[self.model.max_depth] * len(self.model.stages), # Use model's max_depth
                            kernel_sizes=[[7] * self.model.max_depth for _ in self.model.stages], # Use model's max_depth
                            expand_ratios=[[6] * self.model.max_depth for _ in self.model.stages] # Use model's max_depth
                        )
                    else:
                        subnet_cfg = self._sample_subnet_for_phase(phase)
                        subnet_cfg.resolution = data.shape[-1]

                    # Forward
                    output = self.model(data, subnet_cfg)

                    # Loss: CE + KD from teacher
                    ce_loss = F.cross_entropy(output, target)

                    if i == 0:
                        teacher_output = output.detach()
                        loss = ce_loss
                    else:
                        # Knowledge distillation from teacher
                        kd_loss = F.kl_div(
                            F.log_softmax(output / 4, dim=1),
                            F.softmax(teacher_output / 4, dim=1),
                            reduction='batchmean'
                        ) * 16  # temperature^2
                        loss = 0.5 * ce_loss + 0.5 * kd_loss

                    batch_loss += loss / subnets_per_batch

                batch_loss.backward()
                self.optimizer.step()
                total_loss += batch_loss.item()

            # Evaluate
            val_acc = self._evaluate()
            avg_loss = total_loss / len(self.train_loader)

            self.history.append({
                'phase': phase, 'epoch': epoch,
                'loss': avg_loss, 'val_acc': val_acc
            })

            print(f"Epoch {epoch+1}/{epochs}: loss={avg_loss:.4f}, val_acc={100*val_acc:.2f}%")

        return {'final_val_acc': val_acc}

    def _evaluate(self) -> float:
        """
        Evaluate on validation set using random subnets.
        """
        self.model.eval()
        correct = 0
        total = 0

        with torch.no_grad():
            for data, target in self.val_loader:
                # Move data to the correct device
                data, target = data.to(device), target.to(device)

                # Evaluate with a random subnet
                subnet_cfg = self.model.sample_subnet()
                subnet_cfg.resolution = data.shape[-1]

                output = self.model(data, subnet_cfg)
                correct += output.argmax(1).eq(target).sum().item()
                total += target.size(0)

        return correct / total

    def train_full(self, epochs_per_phase: int = 3):
        """
        Run complete progressive shrinking training.
        """
        phases = ['full', 'elastic_kernel', 'elastic_depth', 'elastic_expand']

        results = {}
        for phase in phases:
            results[phase] = self.train_phase(phase, epochs=epochs_per_phase)

        return results

In [29]:
# Prepare data for training

transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

full_train = datasets.CIFAR10('./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10('./data', train=False, transform=transform)

# Small subset for demonstration
train_subset = Subset(full_train, range(2000))
val_subset = Subset(full_train, range(2000, 3000))

train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=64)

print(f"Train: {len(train_subset)}, Val: {len(val_subset)}")

Train: 2000, Val: 1000


In [40]:
# Run progressive shrinking training (abbreviated)

print("=" * 60)
print("PROGRESSIVE SHRINKING TRAINING")
print("=" * 60)

ofa_model = OFAMobileNetV3(num_classes=10)
ofa_model.to(device) # Move model to device immediately after creation

trainer = ProgressiveShrinkingTrainer(ofa_model, train_loader, val_loader)

# Train (abbreviated - full training would take hours)
results = trainer.train_full(epochs_per_phase=2)

PROGRESSIVE SHRINKING TRAINING

Phase: full
  Kernels: [7]
  Depths: [2]
  Expands: [6]
Epoch 1/2: loss=1.3123, val_acc=10.90%
Epoch 2/2: loss=1.1376, val_acc=10.00%

Phase: elastic_kernel
  Kernels: [3, 5, 7]
  Depths: [2]
  Expands: [6]
Epoch 1/2: loss=1.1446, val_acc=10.10%
Epoch 2/2: loss=1.0605, val_acc=10.10%

Phase: elastic_depth
  Kernels: [3, 5, 7]
  Depths: [1, 2]
  Expands: [6]
Epoch 1/2: loss=1.1966, val_acc=9.20%
Epoch 2/2: loss=1.1012, val_acc=9.00%

Phase: elastic_expand
  Kernels: [3, 5, 7]
  Depths: [1, 2]
  Expands: [3, 4, 6]
Epoch 1/2: loss=1.2140, val_acc=14.50%
Epoch 2/2: loss=1.1048, val_acc=12.50%


Subnet Sampling & Evaluation

After training, we can extract any subnet and evaluate its performance without retraining.

In [44]:
class SubnetEvaluator:
    """
    Evaluate subnets extracted from OFA network.
    """

    def __init__(self, ofa_model: OFAMobileNetV3, val_loader: DataLoader):
        self.model = ofa_model
        self.val_loader = val_loader

    def evaluate_subnet(self, config: SubnetConfig) -> Dict:
        """
        Evaluate a specific subnet configuration.
        """
        self.model.eval()
        self.model.to(device)

        correct = 0
        total = 0
        latencies = []

        with torch.no_grad():
            for data, target in self.val_loader:
                # Resize to subnet resolution
                if data.shape[-1] != config.resolution:
                    data = F.interpolate(data, size=config.resolution, mode='bilinear')

                data, target = data.to(device), target.to(device)

                start = time.perf_counter()
                output = self.model(data, config)
                latencies.append(time.perf_counter() - start)

                correct += output.argmax(1).eq(target).sum().item()
                total += target.size(0)

        # Estimate FLOPs and params
        flops, params = self._estimate_cost(config)

        return {
            'accuracy': correct / total,
            'latency_ms': np.mean(latencies) * 1000,
            'flops_m': flops / 1e6,
            'params_m': params / 1e6,
        }

    def _estimate_cost(self, config: SubnetConfig) -> Tuple[int, int]:
        """
        Estimate FLOPs and parameters for subnet.
        """
        total_flops = 0
        total_params = 0

        # Rough estimation based on configuration
        spatial = config.resolution // 2  # After stem
        channels = 16

        stage_configs = [(24, 2), (40, 2), (80, 2), (112, 1), (160, 2)]

        for stage_idx, (out_ch, stride) in enumerate(stage_configs):
            depth = config.depths[stage_idx]

            for block_idx in range(depth):
                ks = config.kernel_sizes[stage_idx][block_idx]
                er = config.expand_ratios[stage_idx][block_idx]

                hidden = int(channels * er) if block_idx == 0 else int(out_ch * er)

                # Expand + depthwise + project
                block_flops = (
                    channels * hidden * spatial * spatial +  # Expand
                    hidden * ks * ks * spatial * spatial +   # DW
                    hidden * out_ch * spatial * spatial      # Project
                )
                block_params = channels * hidden + hidden * ks * ks + hidden * out_ch

                total_flops += block_flops
                total_params += block_params

                if block_idx == 0 and stride == 2:
                    spatial //= 2
                channels = out_ch

        return total_flops, total_params

In [46]:
# Evaluate different subnet configurations

print("=" * 60)
print("SUBNET EVALUATION")
print("=" * 60)

evaluator = SubnetEvaluator(ofa_model, val_loader)

# Define specific configurations to compare
configs = [
    ('Maximum', SubnetConfig(
        resolution=224,
        depths=[2, 2, 2, 2, 2], # Adjusted to max_depth of 2
        kernel_sizes=[[7]*2 for _ in range(5)], # Adjusted for depth 2
        expand_ratios=[[6]*2 for _ in range(5)]  # Adjusted for depth 2
    )),
    ('Medium', SubnetConfig(
        resolution=192,
        depths=[2, 2, 2, 2, 2], # Adjusted to max_depth of 2
        kernel_sizes=[[5]*2 for _ in range(5)], # Adjusted for depth 2
        expand_ratios=[[4]*2 for _ in range(5)]  # Adjusted for depth 2
    )),
    ('Minimum', SubnetConfig(
        resolution=160,
        depths=[2, 2, 2, 2, 2],
        kernel_sizes=[[3]*2 for _ in range(5)],
        expand_ratios=[[3]*2 for _ in range(5)]
    )),
]

print(f"\n{'Config':<12} {'Accuracy':>10} {'Latency':>10} {'FLOPs':>10} {'Params':>10}")
print("-" * 55)

for name, config in configs:
    metrics = evaluator.evaluate_subnet(config)
    print(f"{name:<12} {100*metrics['accuracy']:>9.1f}% {metrics['latency_ms']:>9.2f}ms "
          f"{metrics['flops_m']:>9.1f}M {metrics['params_m']:>9.2f}M")

SUBNET EVALUATION

Config         Accuracy    Latency      FLOPs     Params
-------------------------------------------------------
Maximum           11.2%      8.01ms     396.0M      1.08M
Medium            11.4%     11.28ms     158.9M      0.65M
Minimum            9.0%     11.00ms      70.6M      0.46M


 Accuracy Predictor

To avoid evaluating every subnet, train a predictor to estimate accuracy from architecture encoding.

In [47]:
class AccuracyPredictor(nn.Module):
    """
    Neural network to predict subnet accuracy from configuration.

    Enables fast architecture search without evaluation.
    """

    def __init__(self, num_stages: int = 5, max_depth: int = 4,
                 num_kernel_options: int = 3, num_expand_options: int = 3,
                 hidden_dim: int = 64):
        super().__init__()

        self.num_stages = num_stages
        self.max_depth = max_depth

        # Embeddings
        self.resolution_embed = nn.Embedding(5, hidden_dim)  # 5 resolution options
        self.depth_embed = nn.Embedding(max_depth + 1, hidden_dim)
        self.kernel_embed = nn.Embedding(num_kernel_options, hidden_dim)
        self.expand_embed = nn.Embedding(num_expand_options, hidden_dim)

        # MLP
        input_dim = hidden_dim * (1 + num_stages * (1 + 2 * max_depth))
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim * 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )

    def encode_config(self, config: SubnetConfig) -> Dict[str, torch.Tensor]:
        """
        Encode subnet configuration as tensor.
        """
        # Resolution index (map 160,176,192,208,224 to 0-4)
        res_idx = (config.resolution - 160) // 16
        res_idx = max(0, min(4, res_idx))

        # Depth indices
        depth_indices = config.depths

        # Kernel indices (3→0, 5→1, 7→2)
        kernel_indices = []
        for stage_ks in config.kernel_sizes:
            for ks in stage_ks:
                kernel_indices.append((ks - 3) // 2)
            # Pad to max_depth
            kernel_indices.extend([0] * (self.max_depth - len(stage_ks)))

        # Expand indices (3→0, 4→1, 6→2)
        expand_map = {3: 0, 4: 1, 6: 2}
        expand_indices = []
        for stage_er in config.expand_ratios:
            for er in stage_er:
                expand_indices.append(expand_map.get(int(er), 2))
            expand_indices.extend([0] * (self.max_depth - len(stage_er)))

        return {
            'resolution': torch.tensor([res_idx]),
            'depths': torch.tensor(depth_indices),
            'kernels': torch.tensor(kernel_indices),
            'expands': torch.tensor(expand_indices),
        }

    def forward(self, encoded: Dict[str, torch.Tensor]) -> torch.Tensor:
        """
        Predict accuracy from encoded configuration.
        """
        # Embed each component
        res_emb = self.resolution_embed(encoded['resolution'])
        depth_emb = self.depth_embed(encoded['depths']).flatten()
        kernel_emb = self.kernel_embed(encoded['kernels']).flatten()
        expand_emb = self.expand_embed(encoded['expands']).flatten()

        # Concatenate and predict
        features = torch.cat([res_emb.flatten(), depth_emb, kernel_emb, expand_emb])
        return self.mlp(features.unsqueeze(0)).squeeze()

In [48]:
# Build accuracy predictor training data

print("Building Accuracy Predictor Training Data")
print("=" * 50)

# Collect (config, accuracy) pairs
predictor_data = []

for i in range(30):
    config = ofa_model.sample_subnet()
    metrics = evaluator.evaluate_subnet(config)
    predictor_data.append((config, metrics['accuracy']))

    if (i + 1) % 10 == 0:
        print(f"Collected {i+1} samples")

# Train predictor
print("\nTraining accuracy predictor...")

acc_predictor = AccuracyPredictor()
optimizer = torch.optim.Adam(acc_predictor.parameters(), lr=0.01)

for epoch in range(50):
    total_loss = 0
    for config, acc in predictor_data:
        encoded = acc_predictor.encode_config(config)
        pred = acc_predictor(encoded)
        loss = F.mse_loss(pred, torch.tensor(acc))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}: MSE = {total_loss/len(predictor_data):.6f}")

Building Accuracy Predictor Training Data
Collected 10 samples
Collected 20 samples
Collected 30 samples

Training accuracy predictor...
Epoch 10: MSE = 0.010997
Epoch 20: MSE = 0.010997
Epoch 30: MSE = 0.010997
Epoch 40: MSE = 0.010997
Epoch 50: MSE = 0.010997


Deployment Workflow

Complete workflow: given hardware constraints, find and deploy optimal subnet.

In [49]:
class OFADeployer:
    """
    Deploy optimal subnet for given hardware constraints.
    """

    def __init__(self, ofa_model: OFAMobileNetV3,
                 acc_predictor: AccuracyPredictor,
                 evaluator: SubnetEvaluator):
        self.model = ofa_model
        self.predictor = acc_predictor
        self.evaluator = evaluator

    def find_optimal_subnet(self,
                           max_latency_ms: float = None,
                           max_flops_m: float = None,
                           max_params_m: float = None,
                           num_samples: int = 100) -> Tuple[SubnetConfig, Dict]:
        """
        Find best subnet satisfying constraints.
        """
        print(f"Searching for optimal subnet...")
        print(f"  Constraints: latency<={max_latency_ms}ms, "
              f"FLOPs<={max_flops_m}M, params<={max_params_m}M")

        candidates = []

        for i in range(num_samples):
            config = self.model.sample_subnet()

            # Quick cost estimation
            flops, params = self.evaluator._estimate_cost(config)
            flops_m = flops / 1e6
            params_m = params / 1e6

            # Check constraints
            if max_flops_m and flops_m > max_flops_m:
                continue
            if max_params_m and params_m > max_params_m:
                continue

            # Predict accuracy
            encoded = self.predictor.encode_config(config)
            pred_acc = self.predictor(encoded).item()

            candidates.append({
                'config': config,
                'pred_acc': pred_acc,
                'flops_m': flops_m,
                'params_m': params_m
            })

        if not candidates:
            print("No valid candidates found!")
            return None, None

        # Sort by predicted accuracy
        candidates.sort(key=lambda x: -x['pred_acc'])

        # Verify top candidates
        print(f"\nVerifying top {min(5, len(candidates))} candidates...")

        best_config = None
        best_metrics = None

        for cand in candidates[:5]:
            metrics = self.evaluator.evaluate_subnet(cand['config'])

            # Check latency constraint
            if max_latency_ms and metrics['latency_ms'] > max_latency_ms:
                continue

            if best_metrics is None or metrics['accuracy'] > best_metrics['accuracy']:
                best_config = cand['config']
                best_metrics = metrics

        return best_config, best_metrics

    def export_subnet(self, config: SubnetConfig, path: str = None):
        """
        Export subnet as standalone model.
        """
        print(f"\nSubnet configuration:")
        print(f"  Resolution: {config.resolution}")
        print(f"  Depths: {config.depths}")
        print(f"  Total blocks: {sum(config.depths)}")

        if path:
            print(f"  Exported to: {path}")

In [50]:
# Deploy for different hardware targets

print("=" * 60)
print("DEPLOYMENT FOR DIFFERENT HARDWARE")
print("=" * 60)

deployer = OFADeployer(ofa_model, acc_predictor, evaluator)

# Define hardware targets
targets = [
    ('High-end GPU', {'max_flops_m': 500}),
    ('Mobile GPU', {'max_flops_m': 200, 'max_params_m': 3}),
    ('Edge Device', {'max_flops_m': 100, 'max_params_m': 1.5}),
]

deployment_results = []

for name, constraints in targets:
    print(f"\n{'='*50}")
    print(f"Target: {name}")

    config, metrics = deployer.find_optimal_subnet(**constraints, num_samples=50)

    if config:
        deployment_results.append({
            'target': name,
            'config': config,
            'metrics': metrics
        })

        print(f"\nBest subnet found:")
        print(f"  Accuracy: {100*metrics['accuracy']:.1f}%")
        print(f"  Latency: {metrics['latency_ms']:.2f}ms")
        print(f"  FLOPs: {metrics['flops_m']:.1f}M")
        print(f"  Params: {metrics['params_m']:.2f}M")

        deployer.export_subnet(config)

DEPLOYMENT FOR DIFFERENT HARDWARE

Target: High-end GPU
Searching for optimal subnet...
  Constraints: latency<=Nonems, FLOPs<=500M, params<=NoneM

Verifying top 5 candidates...

Best subnet found:
  Accuracy: 15.4%
  Latency: 6.31ms
  FLOPs: 177.8M
  Params: 0.45M

Subnet configuration:
  Resolution: 224
  Depths: [1, 1, 2, 2, 1]
  Total blocks: 7

Target: Mobile GPU
Searching for optimal subnet...
  Constraints: latency<=Nonems, FLOPs<=200M, params<=3M

Verifying top 5 candidates...

Best subnet found:
  Accuracy: 25.7%
  Latency: 7.62ms
  FLOPs: 134.0M
  Params: 0.56M

Subnet configuration:
  Resolution: 160
  Depths: [2, 2, 2, 1, 2]
  Total blocks: 9

Target: Edge Device
Searching for optimal subnet...
  Constraints: latency<=Nonems, FLOPs<=100M, params<=1.5M

Verifying top 5 candidates...

Best subnet found:
  Accuracy: 11.8%
  Latency: 7.23ms
  FLOPs: 91.6M
  Params: 0.70M

Subnet configuration:
  Resolution: 160
  Depths: [2, 1, 1, 2, 2]
  Total blocks: 8
